# 05 · Le format Parquet

Un analyste veut interroger les 3,95 millions de courses de mars, et vite.
Parquet range les données **par colonne** et garde un résumé à la fin du fichier (le *footer*) : on ne lit que ce dont on a besoin.

In [1]:
from pathlib import Path
import shutil
import time
import pandas as pd
import pyarrow.parquet as pq

DATA = Path("../data")
OUTPUT = Path("../output")
FICHIER = DATA / "yellow_tripdata_2026-03.parquet"

## 1. Regarder dans le fichier sans le lire

`pq.read_metadata` ne lit que le footer : le nombre de lignes, de colonnes et de *row groups* (paquets de lignes).

In [2]:
meta = pq.read_metadata(FICHIER)
print(meta.num_rows, meta.num_row_groups, meta.num_columns)

3952451 4 20


3 952 451 lignes rangées en 4 row groups, 20 colonnes : réponse immédiate, sans charger une seule course.

Le footer contient aussi le **schéma** : le type de chaque colonne.

In [3]:
schema = pq.read_schema(FICHIER)
schema

VendorID: int32
tpep_pickup_datetime: timestamp[us]
tpep_dropoff_datetime: timestamp[us]
passenger_count: int64
trip_distance: double
RatecodeID: int64
store_and_fwd_flag: large_string
PULocationID: int32
DOLocationID: int32
payment_type: int64
fare_amount: double
extra: double
mta_tax: double
tip_amount: double
tolls_amount: double
improvement_surcharge: double
total_amount: double
congestion_surcharge: double
Airport_fee: double
cbd_congestion_fee: double

Entiers (`int32`, `int64`), dates (`timestamp[us]`), décimaux (`double`), texte (`large_string`) : le type est écrit dans le fichier.

Pour chaque colonne de chaque row group, le footer garde aussi le minimum, le maximum et le nombre de valeurs vides.

In [4]:
i = schema.get_field_index("tpep_pickup_datetime")
st = meta.row_group(0).column(i).statistics
print(st.min, st.max, st.null_count)

2026-02-28 23:38:17 2026-03-11 23:00:29 0


Le row group 0 contient les départs du 28 février au 11 mars, sans valeur vide.
Pour une question sur le 25 mars, pyarrow sait qu'il peut sauter ce paquet sans le lire.

## 2. Lire tout le fichier

`%%time`, en première ligne d'une cellule, mesure sa durée : *Wall time* = durée réelle écoulée.

In [5]:
%%time
courses = pd.read_parquet(FICHIER)

CPU times: user 1.29 s, sys: 221 ms, total: 1.51 s
Wall time: 263 ms


Moins d'une seconde pour près de 4 millions de lignes.
Regardons les types reçus par pandas.

In [6]:
courses.dtypes.head(3)

VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
dtype: object

Les dates sont déjà des dates et `VendorID` un entier `int32`, sans rien préciser : le schéma voyage avec les données.
Depuis un CSV (notebook 02), ces dates reviennent en texte.

## 3. Lire seulement 2 colonnes

Pour une distance et un montant moyens, 2 colonnes sur 20 suffisent : on les demande avec `columns=`.

In [7]:
%%time
deux_colonnes = pd.read_parquet(FICHIER, columns=["trip_distance", "total_amount"])
deux_colonnes.shape

CPU times: user 77 ms, sys: 8.86 ms, total: 85.9 ms
Wall time: 33.5 ms


(3952451, 2)

Toutes les lignes, mais une lecture plusieurs fois plus rapide : Parquet n'a lu que 2 colonnes (c'est la *projection*).
Un CSV, lui, doit être lu en entier.

## 4. Filtrer pendant la lecture

`filters=` ne garde que les lignes utiles, ici les courses payées en espèces (`payment_type == 2`).

In [8]:
especes = pd.read_parquet(FICHIER, filters=[("payment_type", "==", 2)])
len(especes)

353541

Seules 353 541 courses sur 3,95 millions arrivent en mémoire.
Comment pyarrow évite-t-il de tout lire ? Regardons le min et le max de `payment_type` dans chaque row group.

In [9]:
i = schema.get_field_index("payment_type")
for numero in range(meta.num_row_groups):
    st = meta.row_group(numero).column(i).statistics
    print("row group", numero, ": min", st.min, "max", st.max)

row group 0 : min 1 max 4
row group 1 : min 1 max 4
row group 2 : min 0 max 4
row group 3 : min 0 max 0


Dans le row group 3, le min et le max valent 0 : aucune course en espèces possible, pyarrow le saute sans le lire (c'est le **saut de blocs**).
Les 3 autres row groups peuvent en contenir : ils sont lus.

## 5. Choisir la compression

Parquet compresse chaque colonne. On écrit le mois avec 4 compressions (`"none"` = aucune) et on compare taille et durée d'écriture. L'écriture gzip est longue : patientez une minute ou deux.

In [10]:
dossier = OUTPUT / "parquet"
dossier.mkdir(parents=True, exist_ok=True)
resultats = []
for compression in ["snappy", "zstd", "gzip", "none"]:
    chemin = dossier / f"courses_{compression}.parquet"
    debut = time.perf_counter()
    courses.to_parquet(chemin, compression=compression, index=False)
    duree = time.perf_counter() - debut
    taille_mo = chemin.stat().st_size / 1_000_000
    resultats.append({"compression": compression, "taille_mo": taille_mo, "ecriture_s": duree})
pd.DataFrame(resultats).round(1)

,compression,taille_mo,ecriture_s
0,snappy,81.8,1.8
1,zstd,66.7,2.1
2,gzip,63.3,38.6
3,none,104.2,1.5


Sans compression : 104 Mo. zstd descend à 67 Mo en s'écrivant presque aussi vite que snappy (le choix par défaut de pandas, 82 Mo).
gzip gagne encore 3 Mo, mais met plus de 10 fois plus de temps à écrire.

## 6. Partitionner : un dossier par *borough*

On ajoute le *borough* de départ grâce à la table des zones.

In [11]:
zones = pd.read_csv(DATA / "taxi_zone_lookup.csv", usecols=["LocationID", "Borough"])
zones.columns = ["PULocationID", "borough"]
courses_borough = courses.merge(zones, on="PULocationID")
courses_borough["borough"].value_counts(dropna=False)

borough
Manhattan        3391948
Queens            353174
Brooklyn          160128
Bronx              39925
Unknown             4624
NaN                 1625
EWR                  527
Staten Island        500
Name: count, dtype: int64

Manhattan concentre l'essentiel des départs (3,4 millions), loin devant le Queens.
1 625 courses n'ont pas de *borough* (`NaN`) : elles partent de la zone 265, hors de New York.

`partition_cols` écrit un sous-dossier par valeur de `borough`. `shutil.rmtree` efface d'abord l'ancien dossier, pour pouvoir relancer la cellule.

In [12]:
lac = OUTPUT / "lake_courses"
shutil.rmtree(lac, ignore_errors=True)
courses_borough.to_parquet(lac, partition_cols=["borough"], index=False)
for sous_dossier in sorted(lac.iterdir()):
    print(sous_dossier.name)

borough=Bronx
borough=Brooklyn
borough=EWR
borough=Manhattan
borough=Queens
borough=Staten%20Island
borough=Unknown
borough=__HIVE_DEFAULT_PARTITION__


Un dossier `borough=...` par valeur ; l'espace de « Staten Island » devient `%20`.
Les courses sans *borough* (`NaN`) vont dans `__HIVE_DEFAULT_PARTITION__`.

Relisons seulement le Queens : pyarrow ne lit qu'un seul sous-dossier.

In [13]:
queens = pd.read_parquet(lac, filters=[("borough", "==", "Queens")])
print(len(queens))
queens.dtypes.tail(3)

353174


Airport_fee            float64
cbd_congestion_fee     float64
borough               category
dtype: object

353 174 courses, le même nombre que dans le comptage plus haut.
La colonne de partition revient en type `category` : sa valeur n'est pas dans les fichiers, mais dans le nom du dossier.

## Exercices

### ✍️ Exercice 1 — Les courses de JFK

Pour les courses parties de l'aéroport JFK (`PULocationID == 132`), lisez seulement `total_amount` et `tip_amount` avec un filtre, puis calculez les deux moyennes.

In [14]:
# ✅ Solution
jfk = pd.read_parquet(FICHIER,
    columns=["total_amount", "tip_amount"],
    filters=[("PULocationID", "==", 132)])
print(len(jfk))
jfk.mean()

153791


total_amount    77.701799
tip_amount       9.198679
dtype: float64

153 791 courses depuis JFK : 77,70 $ en moyenne, dont 9,20 $ de pourboire.
Le filtre peut porter sur une colonne qu'on ne lit pas (`PULocationID`).

### ✍️ Exercice 2 — Quelle compression choisir ?

Mesurez la durée de relecture des 4 fichiers de `output/parquet/`.
Quelle compression choisir pour archiver le mois pendant des années ? Et pour le relire vite et souvent ?

In [15]:
# ✅ Solution
for compression in ["snappy", "zstd", "gzip", "none"]:
    debut = time.perf_counter()
    pd.read_parquet(dossier / f"courses_{compression}.parquet")
    print(compression, round(time.perf_counter() - debut, 2), "s")

snappy 0.27 s


zstd 0.24 s


gzip 0.35 s
none 0.18 s


Les 4 relectures prennent des temps du même ordre : la compression joue surtout sur la taille et sur l'écriture.
Pour archiver, gzip ou zstd (les plus petits) ; pour relire vite et souvent, snappy ou zstd : zstd convient aux deux.

## À retenir

- Parquet range les données par colonne et garde le schéma : les types reviennent intacts.
- Le footer (lignes, schéma, min et max par row group) se lit sans charger les données.
- `columns=` (projection) et `filters=` (saut de blocs) : on ne lit que l'utile.
- zstd : petit et rapide, un bon choix par défaut ; gzip coûte cher à l'écriture.
- `partition_cols` : un dossier par valeur d'une colonne souvent filtrée et peu variée.